# 04 — Factuality & Hallucination Detection

32 examples covering ProvenanceLLM, ProvenanceEmbeddings, GroundedAIHallucination,
BespokeMiniCheck, LLMRAGEvaluator, and WikiProvenance.

**Installation:**
```bash
pip install guardrails-ai openai python-dotenv
guardrails hub install hub://guardrails/provenance_llm
guardrails hub install hub://guardrails/provenance_embeddings
guardrails hub install hub://guardrails/grounded_ai_hallucination
guardrails hub install hub://guardrails/bespoke_minicheck
guardrails hub install hub://guardrails/llm_rag_evaluator
guardrails hub install hub://guardrails/wiki_provenance
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

import openai
from guardrails import Guard, OnFailAction
from guardrails.errors import ValidationError

oai = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
print('Setup complete.')

In [ ]:
# Install Hub validators (run once)
!guardrails hub install hub://guardrails/provenance_llm --quiet
!guardrails hub install hub://guardrails/provenance_embeddings --quiet
!guardrails hub install hub://guardrails/grounded_ai_hallucination --quiet
!guardrails hub install hub://guardrails/bespoke_minicheck --quiet
!guardrails hub install hub://guardrails/llm_rag_evaluator --quiet
!guardrails hub install hub://guardrails/wiki_provenance --quiet

## ProvenanceLLM Examples (01–07)

In [ ]:
# Example 01: Unsupported claim blocked — no source context provided
from guardrails.hub import ProvenanceLlm
guard = Guard().use(
    ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION)
)
try:
    guard.validate(
        'The Eiffel Tower was built in 1650.',
        metadata={'sources': ['The Eiffel Tower is a wrought-iron lattice tower completed in 1889.']}
    )
except ValidationError:
    print('FAIL - claim not grounded in source (wrong year)')

In [ ]:
# Example 02: Grounded claim passes — matches source document
from guardrails.hub import ProvenanceLlm
guard = Guard().use(
    ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION)
)
source = 'The Eiffel Tower is a wrought-iron lattice tower completed in 1889 in Paris, France.'
claim = 'The Eiffel Tower was completed in 1889.'
outcome = guard.validate(claim, metadata={'sources': [source]})
print('PASS - grounded claim:', outcome.validation_passed)

In [ ]:
# Example 03: Partial match — threshold effect
from guardrails.hub import ProvenanceLlm
source = 'Python is a high-level, interpreted programming language known for its readability.'
claim = 'Python is a compiled language designed for web development.'
for threshold in [0.3, 0.6, 0.9]:
    guard = Guard().use(
        ProvenanceLlm(llm_callable='openai/gpt-4o-mini', threshold=threshold, on_fail=OnFailAction.NOOP)
    )
    outcome = guard.validate(claim, metadata={'sources': [source]})
    print(f'  threshold={threshold}  passed={outcome.validation_passed}')

In [ ]:
# Example 04: metadata sources list — proper format for ProvenanceLLM
from guardrails.hub import ProvenanceLlm
guard = Guard().use(
    ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP)
)
sources = [
    'Mars is the fourth planet from the Sun with a thin atmosphere.',
    'Mars has two moons: Phobos and Deimos.',
    'The surface of Mars is red due to iron oxide (rust).'
]
outcome = guard.validate(
    'Mars has two moons named Phobos and Deimos.',
    metadata={'sources': sources}
)
print('multi-source provenance passed:', outcome.validation_passed)

In [ ]:
# Example 05: ProvenanceLLM + REASK — LLM re-prompted to revise ungrounded statement
from guardrails.hub import ProvenanceLlm
guard = Guard().use(
    ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.REASK)
)
source = 'The Amazon River is the largest river by discharge in the world and is located in South America.'
outcome = guard(
    oai.chat.completions.create,
    prompt='Write one sentence about the Amazon River.',
    model=MODEL,
    num_reasks=1,
    metadata={'sources': [source]}
)
print('REASK result:', outcome.validated_output)

In [ ]:
# Example 06: Medical claim scenario — doctor AI must cite source
from guardrails.hub import ProvenanceLlm
guard = Guard().use(
    ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION)
)
drug_source = 'Metformin is typically prescribed at 500-2000 mg/day for type 2 diabetes.'
hallucinated = 'Metformin is prescribed at 5000 mg/day for type 2 diabetes.'
try:
    guard.validate(hallucinated, metadata={'sources': [drug_source]})
except ValidationError:
    print('FAIL - medical claim contradicts source (wrong dosage)')

In [ ]:
# Example 07: Financial earnings claim not in source — blocked
from guardrails.hub import ProvenanceLlm
guard = Guard().use(
    ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION)
)
report = 'Company XYZ reported Q3 revenue of $1.2 billion, up 15% year-over-year.'
wrong_claim = 'Company XYZ reported Q3 revenue of $3 billion.'
try:
    guard.validate(wrong_claim, metadata={'sources': [report]})
except ValidationError:
    print('FAIL - financial figure contradicts source')

## ProvenanceEmbeddings Examples (08–10)

In [ ]:
# Example 08: Cosine similarity threshold — embedding-based grounding
from guardrails.hub import ProvenanceEmbeddings
guard = Guard().use(
    ProvenanceEmbeddings(threshold=0.7, on_fail=OnFailAction.NOOP)
)
source = 'The speed of light in vacuum is approximately 299,792,458 meters per second.'
claim = 'Light travels at about 300,000 km/s in a vacuum.'
outcome = guard.validate(claim, metadata={'sources': [source]})
print('embedding similarity passed:', outcome.validation_passed)

In [ ]:
# Example 09: High semantic similarity — paraphrase of source text passes
from guardrails.hub import ProvenanceEmbeddings
guard = Guard().use(
    ProvenanceEmbeddings(threshold=0.6, on_fail=OnFailAction.EXCEPTION)
)
source = 'Dogs are domesticated mammals known for their loyalty and companionship.'
paraphrase = 'Dogs are tame animals famous for being loyal companions to humans.'
outcome = guard.validate(paraphrase, metadata={'sources': [source]})
print('high-similarity paraphrase passed:', outcome.validation_passed)

In [ ]:
# Example 10: Low semantic similarity — completely different claim fails
from guardrails.hub import ProvenanceEmbeddings
guard = Guard().use(
    ProvenanceEmbeddings(threshold=0.8, on_fail=OnFailAction.EXCEPTION)
)
source = 'Dogs are domesticated mammals known for their loyalty.'
unrelated_claim = 'The stock market reached an all-time high in January 2024.'
try:
    guard.validate(unrelated_claim, metadata={'sources': [source]})
except ValidationError:
    print('FAIL - claim has low semantic similarity to source')

## GroundedAIHallucination Examples (11–14)

In [ ]:
# Example 11: Summary contains invented fact — hallucination detected
from guardrails.hub import GroundedAIHallucination
guard = Guard().use(
    GroundedAIHallucination(on_fail=OnFailAction.EXCEPTION)
)
article = 'The annual conference attracted 500 attendees from 20 countries.'
bad_summary = 'The conference had 2,000 attendees from over 50 countries and won an award.'
try:
    guard.validate(bad_summary, metadata={'sources': [article]})
except ValidationError:
    print('FAIL - hallucinated facts detected in summary')

In [ ]:
# Example 12: Faithful summary passes GroundedAIHallucination
from guardrails.hub import GroundedAIHallucination
guard = Guard().use(
    GroundedAIHallucination(on_fail=OnFailAction.EXCEPTION)
)
article = 'The annual conference attracted 500 attendees from 20 countries.'
good_summary = 'The conference drew 500 participants from 20 different countries.'
outcome = guard.validate(good_summary, metadata={'sources': [article]})
print('PASS - faithful summary:', outcome.validation_passed)

In [ ]:
# Example 13: Customer service chatbot answer vs FAQ source
from guardrails.hub import GroundedAIHallucination
guard = Guard().use(
    GroundedAIHallucination(on_fail=OnFailAction.EXCEPTION)
)
faq = 'Returns are accepted within 30 days of purchase with original receipt.'
bot_response = 'You can return items within 60 days without any receipt needed.'
try:
    guard.validate(bot_response, metadata={'sources': [faq]})
except ValidationError:
    print('FAIL - chatbot hallucinated wrong return policy')

In [ ]:
# Example 14: News summarization — model adds detail not in original article
from guardrails.hub import GroundedAIHallucination
guard = Guard().use(
    GroundedAIHallucination(on_fail=OnFailAction.EXCEPTION)
)
article = 'A fire broke out at a warehouse in Chicago on Tuesday morning. No injuries were reported.'
hallucinated_summary = 'A fire at a Chicago warehouse caused $2M in damages and injured three workers.'
try:
    guard.validate(hallucinated_summary, metadata={'sources': [article]})
except ValidationError:
    print('FAIL - hallucinated details not in original article')

## BespokeMiniCheck Examples (15–18)

In [ ]:
# Example 15: Unsupported claim contradicts source document
from guardrails.hub import BespokeMinicheck
guard = Guard().use(
    BespokeMinicheck(on_fail=OnFailAction.EXCEPTION)
)
astronomy_doc = 'The Moon orbits Earth and has no significant atmosphere. It was formed approximately 4.5 billion years ago.'
bad_claim = 'The Moon is made of cheese and orbits Mars.'
try:
    guard.validate(bad_claim, metadata={'sources': [astronomy_doc]})
except ValidationError:
    print('FAIL - unsupported claim detected by BespokeMiniCheck')

In [ ]:
# Example 16: Supported historical claim — document confirms fact
from guardrails.hub import BespokeMinicheck
guard = Guard().use(
    BespokeMinicheck(on_fail=OnFailAction.EXCEPTION)
)
history = 'World War II ended in 1945 with the surrender of Germany in May and Japan in September.'
claim = 'World War II concluded in 1945.'
outcome = guard.validate(claim, metadata={'sources': [history]})
print('PASS - supported historical claim:', outcome.validation_passed)

In [ ]:
# Example 17: Mixed-evidence document — partial support scenario
from guardrails.hub import BespokeMinicheck
guard = Guard().use(
    BespokeMinicheck(on_fail=OnFailAction.NOOP)
)
doc = 'Python was created by Guido van Rossum and released in 1991. It is dynamically typed.'
mixed_claim = 'Python was created in 1991 and is statically typed.'
outcome = guard.validate(mixed_claim, metadata={'sources': [doc]})
print('mixed evidence result (passed=%s)' % outcome.validation_passed)

In [ ]:
# Example 18: Legal document summary verification
from guardrails.hub import BespokeMinicheck
guard = Guard().use(
    BespokeMinicheck(on_fail=OnFailAction.EXCEPTION)
)
contract = 'The contractor shall deliver the software within 90 days of contract signing.'
good_summary = 'The delivery timeline is 90 days from when the contract is signed.'
outcome = guard.validate(good_summary, metadata={'sources': [contract]})
print('PASS - accurate contract summary:', outcome.validation_passed)

## LLMRAGEvaluator Examples (19–23)

In [ ]:
# Example 19: RAG answer correctly references retrieved documents
from guardrails.hub import LlmRagEvaluator
guard = Guard().use(
    LlmRagEvaluator(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP)
)
chunks = ['Guardrails AI provides output validation for LLMs. It supports 70+ validators.']
answer = 'Guardrails AI helps validate LLM outputs and has over 70 available validators.'
outcome = guard.validate(answer, metadata={'sources': chunks, 'query': 'What does Guardrails AI do?'})
print('RAG answer grounded:', outcome.validation_passed)

In [ ]:
# Example 20: Hallucinated RAG answer — detail not in retrieved chunks
from guardrails.hub import LlmRagEvaluator
guard = Guard().use(
    LlmRagEvaluator(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION)
)
chunks = ['Guardrails AI provides output validation for LLMs.']
hallucinated = 'Guardrails AI was founded in 2010 and has 500 enterprise customers in Europe.'
try:
    guard.validate(hallucinated, metadata={'sources': chunks, 'query': 'Tell me about Guardrails AI.'})
except ValidationError:
    print('FAIL - RAG answer contains hallucinated details')

In [ ]:
# Example 21: Multi-chunk retrieval metadata scenario
from guardrails.hub import LlmRagEvaluator
guard = Guard().use(
    LlmRagEvaluator(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP)
)
chunks = [
    'Photosynthesis converts light energy into chemical energy stored as glucose.',
    'Chlorophyll in plant cells absorbs sunlight to drive photosynthesis.',
    'The process produces oxygen as a byproduct.'
]
answer = 'Photosynthesis uses chlorophyll to convert sunlight into glucose and releases oxygen.'
outcome = guard.validate(answer, metadata={'sources': chunks, 'query': 'Explain photosynthesis.'})
print('multi-chunk RAG answer grounded:', outcome.validation_passed)

In [ ]:
# Example 22: E-commerce RAG — product description grounded in spec sheet
from guardrails.hub import LlmRagEvaluator
guard = Guard().use(
    LlmRagEvaluator(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.EXCEPTION)
)
spec = 'ProWidget X: 12V DC motor, 500W peak power, 5-year warranty, IP67 waterproof.'
bad_desc = 'ProWidget X features a 24V motor, 1000W power, with a lifetime warranty.'
try:
    guard.validate(bad_desc, metadata={'sources': [spec], 'query': 'Describe the ProWidget X.'})
except ValidationError:
    print('FAIL - product description contradicts spec sheet')

In [ ]:
# Example 23: LLMRAGEvaluator + FILTER — removes hallucinated sentences
from guardrails.hub import LlmRagEvaluator
guard = Guard().use(
    LlmRagEvaluator(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.FILTER)
)
chunks = ['Python is an interpreted language with dynamic typing.']
mixed = 'Python is an interpreted language. It was invented in 1850 by Charles Babbage.'
outcome = guard.validate(mixed, metadata={'sources': chunks, 'query': 'What is Python?'})
print('FILTER result:', outcome.validated_output)

## WikiProvenance Examples (24–25)

In [ ]:
# Example 24: Wikipedia-grounded claim — well-known fact supported
from guardrails.hub import WikiProvenance
guard = Guard().use(
    WikiProvenance(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP)
)
claim = 'The Great Wall of China was built over many centuries to protect against invasions.'
outcome = guard.validate(claim)
print('wiki-grounded claim result:', outcome.validation_passed)

In [ ]:
# Example 25: Unsupported obscure claim not verifiable via Wikipedia
from guardrails.hub import WikiProvenance
guard = Guard().use(
    WikiProvenance(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP)
)
fabricated = 'The fictional city of Zorbania was founded in 3000 BCE and had a population of 50 million.'
outcome = guard.validate(fabricated)
print('unverifiable claim passed:', outcome.validation_passed)  # should fail

## Combined & Advanced Examples (26–32)

In [ ]:
# Example 26: Dual-layer hallucination guard — ProvenanceLLM + BespokeMiniCheck
from guardrails.hub import ProvenanceLlm, BespokeMinicheck
guard = (
    Guard()
    .use(ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP))
    .use(BespokeMinicheck(on_fail=OnFailAction.NOOP))
)
source = 'The Sun is a star at the center of our solar system, approximately 4.6 billion years old.'
outcome = guard.validate(
    'The Sun is about 4.6 billion years old.',
    metadata={'sources': [source]}
)
print('dual hallucination check passed:', outcome.validation_passed)

In [ ]:
# Example 27: Pydantic model with summary field guarded by ProvenanceLLM
from pydantic import BaseModel
from guardrails.hub import ProvenanceLlm

class ResearchSummary(BaseModel):
    topic: str
    summary: str

guard = Guard.for_pydantic(output_class=ResearchSummary)
result = guard(
    oai.chat.completions.create,
    prompt='Return a JSON with topic and summary about black holes.',
    model=MODEL
)
print('structured output:', result.validated_output)

In [ ]:
# Example 28: Threshold sweep — same claim at 0.3, 0.5, 0.7
from guardrails.hub import ProvenanceLlm
source = 'Jupiter is the largest planet in the solar system and has at least 95 known moons.'
claim = 'Jupiter is the biggest planet and has many moons.'
for threshold in [0.3, 0.5, 0.7]:
    guard = Guard().use(
        ProvenanceLlm(llm_callable='openai/gpt-4o-mini', threshold=threshold, on_fail=OnFailAction.NOOP)
    )
    outcome = guard.validate(claim, metadata={'sources': [source]})
    print(f'  threshold={threshold}  passed={outcome.validation_passed}')

In [ ]:
# Example 29: ProvenanceLLM vs ProvenanceEmbeddings — same test, different approaches
from guardrails.hub import ProvenanceLlm, ProvenanceEmbeddings
source = 'Water boils at 100 degrees Celsius at sea level.'
claim = 'Water reaches its boiling point at 100°C under standard atmospheric pressure.'

for name, validator in [
    ('ProvenanceLLM', ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP)),
    ('ProvenanceEmb', ProvenanceEmbeddings(threshold=0.7, on_fail=OnFailAction.NOOP)),
]:
    guard = Guard().use(validator)
    outcome = guard.validate(claim, metadata={'sources': [source]})
    print(f'  {name}: passed={outcome.validation_passed}')

In [ ]:
# Example 30: Medical Q&A bot with source enforcement — end-to-end
from guardrails.hub import ProvenanceLlm
guard = Guard().use(
    ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.REASK)
)
medical_ref = 'Aspirin 325mg can reduce fever and pain. It should not be given to children under 12.'
outcome = guard(
    oai.chat.completions.create,
    prompt='What is the recommended aspirin dosage for adults?',
    model=MODEL,
    num_reasks=1,
    metadata={'sources': [medical_ref]}
)
print('medical Q&A output:', outcome.validated_output)

In [ ]:
# Example 31: Factuality REASK loop — iterate until grounded answer produced
from guardrails.hub import ProvenanceLlm
guard = Guard().use(
    ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.REASK)
)
source = 'The Nile River is approximately 6,650 km long and is the longest river in Africa.'
outcome = guard(
    oai.chat.completions.create,
    prompt='How long is the Nile River?',
    model=MODEL,
    num_reasks=2,
    metadata={'sources': [source]}
)
print('grounded answer:', outcome.validated_output)
print('total iterations:', len(guard.history[0].iterations))

In [ ]:
# Example 32: Multi-document provenance — any-match logic
from guardrails.hub import ProvenanceLlm
guard = Guard().use(
    ProvenanceLlm(llm_callable='openai/gpt-4o-mini', on_fail=OnFailAction.NOOP)
)
sources = [
    'DNA carries genetic information in the form of nucleotide sequences.',
    'Genes are segments of DNA that code for specific proteins.',
    'Mutations are changes in DNA sequences that can affect gene expression.'
]
claim = 'Genes are portions of DNA that encode proteins.'
outcome = guard.validate(claim, metadata={'sources': sources})
print('multi-doc any-match result:', outcome.validation_passed)